# Admin Cashless Adoption
Logistic Regression — Classify whether a transaction will be cashless.

Run cells top-to-bottom: **Setup → Generate Data → Train → Evaluate → Test**

## 1. Setup
Resolve paths for `data/`, `output/`, `report/` directories.

In [ ]:
from pathlib import Path

try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path().resolve()

BASE_DIR   = NOTEBOOK_DIR.parent
DATA_DIR   = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
REPORT_DIR = BASE_DIR / "report"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR   : {BASE_DIR}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"REPORT_DIR : {REPORT_DIR}")

## 2. Generate Data
Create synthetic training data and save to `data/`.

In [ ]:
import csv
import random
import math
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
DATA_DIR = SCRIPT_DIR.parent / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42)

def normal_noise(mu=0, sigma=1):
    u1 = random.random()
    u2 = random.random()
    z = math.sqrt(-2 * math.log(u1)) * math.cos(2 * math.pi * u2)
    return mu + sigma * z

# category_encoded: 0=Hot Food, 1=Household Items, 2=Trendy Food,
#                   3=Desserts, 4=Games, 5=Drinks, 6=Snacks
rows = []
for _ in range(2000):
    category_encoded = random.randint(0, 6)
    amount_bucket = random.randint(0, 4)
    hour_encoded = random.randint(0, 5)
    is_weekend = random.randint(0, 1)
    logit = (-0.5
             + 0.4 * hour_encoded
             + 0.3 * amount_bucket
             + 0.8 * is_weekend
             - 0.2 * category_encoded
             + normal_noise(0, 0.5))
    is_cashless = 1 if logit > 0 else 0
    rows.append({
        "category_encoded": category_encoded,
        "amount_bucket": amount_bucket,
        "hour_encoded": hour_encoded,
        "is_weekend": is_weekend,
        "is_cashless": is_cashless
    })

out_path = DATA_DIR / "cashless_data.csv"
with open(out_path, "w", newline="") as f:
    writer = csv.DictWriter(
        f, fieldnames=["category_encoded", "amount_bucket", "hour_encoded", "is_weekend", "is_cashless"]
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} rows to {out_path}")

## 3. Train
Fit the model and save to `output/`.

In [ ]:
import csv
import pickle
import math
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
DATA_DIR = SCRIPT_DIR.parent / "data"
OUTPUT_DIR = SCRIPT_DIR.parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def sigmoid(z):
    return 1 / (1 + math.exp(-max(-500, min(500, z))))

def fit_logistic(X, y, lr=0.1, epochs=200):
    Xb = [[1] + row for row in X]
    w = [0.0] * len(Xb[0])
    n = len(y)
    for _ in range(epochs):
        grad = [0.0] * len(w)
        for xi, yi in zip(Xb, y):
            pred = sigmoid(sum(w[j] * xi[j] for j in range(len(w))))
            err = pred - yi
            for j in range(len(w)):
                grad[j] += err * xi[j]
        w = [w[j] - lr * grad[j] / n for j in range(len(w))]
    return w

def predict_logistic(w, X):
    Xb = [[1] + row for row in X]
    return [1 if sigmoid(sum(w[j] * xi[j] for j in range(len(w)))) >= 0.5 else 0 for xi in Xb]

# ---------- Load data ----------
rows = []
with open(DATA_DIR / "cashless_data.csv", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

X_all = [
    [int(r["category_encoded"]), int(r["amount_bucket"]), int(r["hour_encoded"]), int(r["is_weekend"])]
    for r in rows
]
y_all = [int(r["is_cashless"]) for r in rows]

split = int(0.8 * len(X_all))
X_train, X_test = X_all[:split], X_all[split:]
y_train, y_test = y_all[:split], y_all[split:]

w = fit_logistic(X_train, y_train, lr=0.1, epochs=200)

model = {
    "weights": w,
    "feature_cols": ["category_encoded", "amount_bucket", "hour_encoded", "is_weekend"],
    "X_test": X_test,
    "y_test": y_test,
    "all_rows": rows
}

out_path = OUTPUT_DIR / "cashless_model.pkl"
with open(out_path, "wb") as f:
    pickle.dump(model, f)

print(f"Model saved to {out_path}")
print(f"Weights: {[round(ww, 4) for ww in w]}")

## 4. Evaluate
Compute metrics on the test set and save to `report/`.

In [ ]:
import pickle
import json
import math
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
OUTPUT_DIR = SCRIPT_DIR.parent / "output"
REPORT_DIR = SCRIPT_DIR.parent / "report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def sigmoid(z):
    return 1 / (1 + math.exp(-max(-500, min(500, z))))

def predict_logistic(w, X):
    Xb = [[1] + row for row in X]
    return [1 if sigmoid(sum(w[j] * xi[j] for j in range(len(w)))) >= 0.5 else 0 for xi in Xb]

def predict_prob(w, X):
    Xb = [[1] + row for row in X]
    return [sigmoid(sum(w[j] * xi[j] for j in range(len(w)))) for xi in Xb]

with open(OUTPUT_DIR / "cashless_model.pkl", "rb") as f:
    model = pickle.load(f)

w = model["weights"]
X_test = model["X_test"]
y_test = model["y_test"]
all_rows = model["all_rows"]

y_pred = predict_logistic(w, X_test)

# Metrics
tp = sum(1 for t, p in zip(y_test, y_pred) if t == 1 and p == 1)
tn = sum(1 for t, p in zip(y_test, y_pred) if t == 0 and p == 0)
fp = sum(1 for t, p in zip(y_test, y_pred) if t == 0 and p == 1)
fn = sum(1 for t, p in zip(y_test, y_pred) if t == 1 and p == 0)

accuracy = round((tp + tn) / len(y_test), 4)
precision = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0.0
recall = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0.0
f1 = round(2 * precision * recall / (precision + recall), 4) if (precision + recall) > 0 else 0.0

# Cashless rate per category
category_names = ["Hot Food", "Household Items", "Trendy Food", "Desserts", "Games", "Drinks", "Snacks"]
all_X = [
    [int(r["category_encoded"]), int(r["amount_bucket"]), int(r["hour_encoded"]), int(r["is_weekend"])]
    for r in all_rows
]
all_y_true = [int(r["is_cashless"]) for r in all_rows]
all_y_pred = predict_logistic(w, all_X)

cashless_by_category = []
for cat_id, cat_name in enumerate(category_names):
    indices = [i for i, r in enumerate(all_rows) if int(r["category_encoded"]) == cat_id]
    if not indices:
        continue
    actual_pct = round(sum(all_y_true[i] for i in indices) / len(indices) * 100, 1)
    pred_pct = round(sum(all_y_pred[i] for i in indices) / len(indices) * 100, 1)
    cashless_by_category.append({
        "category": cat_name,
        "current_pct": actual_pct,
        "predicted_pct": pred_pct
    })

report = {
    "model": "Logistic Regression",
    "task": "Cashless payment prediction",
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "cashless_by_category": cashless_by_category
}

out_path = REPORT_DIR / "evaluation_report.json"
with open(out_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Report saved to {out_path}")
print(f"Accuracy={accuracy}  Precision={precision}  Recall={recall}  F1={f1}")

## 5. Test
Run inference on sample inputs.

In [ ]:
import pickle
import math
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
OUTPUT_DIR = SCRIPT_DIR.parent / "output"

def sigmoid(z):
    return 1 / (1 + math.exp(-max(-500, min(500, z))))

def predict_prob(w, X):
    Xb = [[1] + row for row in X]
    return [sigmoid(sum(w[j] * xi[j] for j in range(len(w)))) for xi in Xb]

with open(OUTPUT_DIR / "cashless_model.pkl", "rb") as f:
    model = pickle.load(f)

w = model["weights"]

category_names = ["Hot Food", "Household Items", "Trendy Food", "Desserts", "Games", "Drinks", "Snacks"]

# Fixed: hour=3, amount=2, weekend=1
# Features order: [category_encoded, amount_bucket, hour_encoded, is_weekend]
sample_X = [[cat_id, 2, 3, 1] for cat_id in range(7)]
probs = predict_prob(w, sample_X)

print("\n=== Cashless Adoption Probability by Category (hour=9PM, amount_bucket=2, weekend) ===")
print(f"{'Category':<20} {'Cashless Prob':>14} {'Prediction':>12}")
for name, prob in zip(category_names, probs):
    label = "CASHLESS" if prob >= 0.5 else "CASH"
    print(f"{name:<20} {prob:>13.1%} {label:>12}")